
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# 데모 - Agent Bricks와 함께 지식 어시스턴트(KA) 에이전트 구축

## 개요

이번 데모에서는 **Agent Bricks**를 사용해 고품질 **지식 어시스턴트** 에이전트를 만드는 방법을 살펴보겠습니다: 지식 어시스턴트. Databricks의 선언적 에이전트 생성 도구를 사용해 문서를 기반으로 질문과 답변 챗봇을 만드는 과정을 안내하고, 전문가의 피드백과 라벨링 세션을 통해 품질을 향상시킬 것입니다.

**시나리오:** **Orion Knowledge Assistant (OKA)** 는 내부 설계 매뉴얼, 규정 준수 문서, Databricks에 저장된 유지보수 가이드에서 얻은 즉각적이고 맥락에 기반한 답변을 제공함으로써 Orion A1 휴머노이드 플랫폼에서 작업하는 엔지니어와 기술자를 지원합니다. 현장 엔지니어가 모션 컨트롤러를 재보정하거나 펌웨어 체크섬을 검증하는 방법을 묻면, OKA는 정확한 절차를 찾아와 올바른 섹션을 참조하거나, 정보가 없는 경우 명확히 알려줍니다. 이 접근법은 임무 중요 환경에서 신뢰와 정확성을 유지합니다.


## 학습 목표
- Agent Bricks로 지식 어시스턴트 에이전트를 만들기 위한 키 구성 요소와 요구사항을 **식별**하세요.
- 지식 소스로 Unity Catalog 파일을 사용하여 지식 보조 에이전트를 **구성**하고 생성합니다.
- 구현 AI Playground을 사용하여 에이전트 테스트 및 평가를 구현하며, 인용 및 소스 검증을 **포함**합니다.
- 라벨링 세션과 전문가 피드백 수집을 통해 에이전트의 질을 **향상시키**세요.
- 에이전트 최적화 및 성능 모니터링을 위한 모범 사례를 **적용**하십시오.

## 요구 사항
- Mosaic AI Agent Bricks 미리보기 (베타)가 활성화된 워크스페이스.
- **서버리스 Compute (환경 버전 5)**. [여기](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version)에 따라 적절한 환경 버전을 선택하세요. 


## 준비

아래 코드를 실행하여 필요한 라이브러리를 설치하고 교실 환경을 구성하세요.

이 단계는 모든 의존성이 사용 가능하고 워크스페이스가 데모 준비가 완료되도록 보장합니다.


In [0]:
%run ../Includes/Classroom-Setup-05

   
## A. KA 에이전트 생성

**에이전트 이름:** **Orion Knowledge Assistant(OKA)**

문서들은 Unity Catalog(UC) 볼륨에 보관됩니다. 저희는 에이전트 브릭스를 사용해 KA 에이전트를 만들 것입니다.

**에이전트를 만드는 단계:**

1. Databricks workspace에서 왼쪽 내비게이션 창의 **에이전트**로 이동하세요
1. **Create Agent** 버튼을 클릭합니다.
1. **Knowledge Assistant** 상자를 클릭하여 새 KA 에이전트 생성을 시작합니다.
1. 다음 정보를 입력합니다:
   - **이름**: `Orion_Knowledge_Assistant`
   - **설명**: 다음과 같은 설명을 입력하세요:  
     `Orion Knowledge Assistant (OKA) helps engineers and technicians quickly find accurate answers from Orion's internal manuals, maintenance guides, and safety documents. It delivers clear, verified responses with source references, reducing search time and ensuring consistent, reliable information across teams.`
   - **정보 소스 유형(Knowledge source type)**: 볼륨 내의 파일
   - **출처**: 사용한 카탈로그에 따라 Unity Catalog 권을 선택하세요(권 이름은 위 참조).
   - **확인 (Confirm)** 버튼을 클릭하세요
   - **지식 소스 이름**: `Company Documents`
   - **콘텐츠 설명**: 다음과 같은 콘텐츠 설명을 입력하세요:  
     `Contains Orion technical documentation, engineering notes, handbooks, and frequently asked questions.`
1. *(선택)* 에이전트가 어떻게 반응해야 하는지에 대한 지침을 추가하세요. 아래 예시 지침을 참고하세요.
1. 생성 과정을 시작하려면 **에이전트 생성 (Create Agent)** 를 클릭하세요

샘플 지시 프롬프트:  
`You are the Orion Knowledge Assistant (OKA). Respond in a clear, professional, and factual tone appropriate for engineers and technical staff. Use only verified information from Orion's internal documents, and include source references when available. If the answer cannot be found, clearly state that and suggest related sections or next steps. Do not speculate, make assumptions, or provide information outside the provided context.`

**⏳ 참고:** 에이전트를 만들고 지식 소스를 동기화하는 데 최대 10분이 걸릴 수 있습니다.

## B. 에이전트 테스트

에이전트가 빌드를 마친 후에는 내장 채팅 인터페이스를 사용해 기능을 테스트할 수 있습니다. 채팅 영역은 Agent Bricks 인터페이스 오른쪽에 나타납니다.

**시험할 샘플 문제들:**

샘플 문서를 바탕으로 에이전트에게 다음과 같은 질문을 해보세요:

1. "오리온은 ISO 13849-1 규정 준수를 어떻게 검증하나요?"
1. "오리온 모션 컨트롤러는 고속 이동 중에 어떻게 안정성을 유지하나요?"
1. **"오리온의 빨간 깜빡이는 불빛은 무엇을 의미하나요?"** 
*참고: 오리온 문서에는 빨간 깜빡이는 불빛이 없습니다.* 

**주의할 점:**
- 문서에 근거한 정확한 답변
- 적절한 인용 및 출처 참고
- 지식 기반 외의 질문에 대한 적절한 처리
- 전문적이고 도움이 되는 어조

**💡 마지막 질문에 대해 에이전트가 더 나은 답변을 받으려면 에이전트의 품질을 개선해야 합니다. 다음으로 그 부분으로 넘어가죠!**

   
## C. 에이전트 품질 향상

Agent Bricks: Knowledge Assistant는 전문가의 피드백과 라벨링 세션을 통해 에이전트의 품질을 향상시킬 수 있게 해줍니다. 이 기능들은 주제 전문가로부터 자연어 피드백을 수집하고 이를 활용해 에이전트의 성능을 재훈련하고 최적화할 수 있게 합니다.

**품질 개선을 언제 사용할지:**
- 에이전트가 부정확하거나 불완전한 답변을 제공할 때
- 에이전트의 어조와 의사소통 스타일을 미세 조정하기
- 새로운 지식 영역이나 사용 사례로 확장할 때
- 사용자 피드백을 기반으로 한 지속적인 최적화
- 다양한 문제 유형에서 일관된 성능을 보장하기





### C1. 품질 향상을 위한 레이블 데이터 활용

이 단계에서는 위의 질문에 대한 에이전트의 응답을 개선하기 위해 레이블 데이터를 제공할 것입니다.

**품질 개선 과정:**
1. 에이전트 인터페이스의 **예시 (Examples)** 탭으로 이동하세요.
1. **추가 (Add)** 버튼을 클릭하세요.
1. 질문(`What does the red blinking light on Orion mean?`)을 입력하고 **추가 (Add)** 를 눌러 질문을 추가하세요.
1. 질문을 클릭하면 세부 정보를 열어보세요.
1. 다음과 같은 **지침 (Guidelines)** 을 추가하세요:
  
    >오리온에 깜빡이는 빨간 불빛이 없다는 것을 사용자에게 알려주세요

    >사용자에게 배터리를 분리했다가 다시 넣어 Orion을 재시작하도록 요청하세요
      
    >밝은 색깔을 확인하세요
      
    >불빛이 깜빡이는지 다시 한 번 확인하세요
  

에이전트의 다양한 지식을 평가하는 평가 질문을 추가하세요.

**에이전트를 테스트하세요:**
1. 에이전트에게 다시 돌아가 같은 질문을 하여 **응답이 개선되었는지 확인하십시오**. 또한 질문을 **다른 색상에 대해 질문하여 답변이 어떻게 적응하는지** 확인하십시오로 바꿀 수도 있습니다. 

### C2. 품질 향상을 위한 전문가 리뷰 활용

**라벨링 세션 및 전문가 검토:**

라벨링 세션 기능은 전문가들이 다음을 가능하게 합니다:
- 평가 질문에 대한 에이전트의 답변 검토합니다
- 응답 품질에 대한 자연어 피드백 제공합니다
- 에이전트 행동에 대한 지침과 기대치 추가합니다
- 응답의 정확성, 완전성, 어조를 평가합니다

전문가 검토 프로세스에 대한 단계별 지침은 [Agent Bricks: 지식 어시스턴트 문서 – 3단계: 품질 개선](https://docs.databricks.com/aws/en/generative-ai/agent-bricks/knowledge-assistant#step-3-improve-quality)을 참조하십시오.

**모범 사례:** 품질 개선은 반복적인 과정입니다. 최적의 에이전트 성과를 달성하기 위해 여러 차례의 피드백 수집 및 개선을 계획하세요.

**💡 질문:** 품질 향상을 위해 지침은 어떻게 사용되나요? 이와 관련해 에이전트 흔적에서 어떤 점을 관찰하셨나요?

   
## D. 자원 정리

이 데모를 완료한 후에는 리소스를 정리하여 불필요한 비용을 피하는 것이 중요합니다. **KA 에이전트는 백그라운드에서 AI Search 엔드포인트와 서빙 엔드포인트를 생성하여 문서에 대한 의미 검색을 가능하게 합니다.** 이 엔드포인트들은 사용 중이 아니더라도 계속 요금이 부과됩니다.

**에이전트를 삭제하려면:**
1. 왼쪽 내비게이션 창에서 **Agents**로 이동하세요
1. **Orion_Knowledge_Assistant** 에이전트를 찾아보세요
1. 에이전트를 클릭하면 세부 정보를 열어보세요
1. 에이전트 옵션 메뉴(오른쪽 상단)에서 **삭제 (Delete)** 를 선택하세요.
1. 삭제하라는 메시지가 나타나면 확인하세요.

에이전트를 삭제하면 관련 AI Search 엔드포인트과 서빙 엔드포인트도 제거됩니다.

   
## 요약

당신은 Agent Bricks을 사용해 KA 에이전트를 성공적으로 구축하고 개선하셨습니다: 지식 보조. 우리가 이룬 성과를 간략히 요약해 드리겠습니다:

**우리가 만든 것:**
- 선언형 Agent Bricks 인터페이스를 사용하여 KA 에이전트를 생성합니다
- Unity Catalog 파일을 지식 소스로 에이전트 구성합니다
- 내장된 채팅 인터페이스를 사용해 에이전트를 테스트했습니다
- 전문가 피드백과 라벨링 세션을 통한 에이전트 품질 향상합니다

**다음 단계 (선택 사항):**

이제 KA 에이전트가 일하고 있다면, 다음 단계를 고려해 보세요:

1. **지식 출처 확장**: 에이전트의 범위를 넓히기 위해 추가 문서 유형과 데이터 소스를 추가하세요
1. **품질 최적화**: 라벨링 세션과 전문가 검토를 통해 에이전트의 성과를 지속적으로 개선하세요
1. **프로덕션 배포**: 적절한 거버넌스와 모니터링을 통해 프로덕션 환경에 에이전트를 배포하십시오.

**추가 자료:**
- [Agent Bricks 문서](https://docs.databricks.com/aws/en/generative-ai/agent-bricks/knowledge-assistant)


&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>